# 02. Training, Evaluation, Ablations & TFLite Export

Trains the NoNooze custom from-scratch 4-block CNN (M1) and baseline models:
- Person-level train/validation/test splits (Zero subject leakage)
- Augmentation pipeline (CLAHE, gamma, noise, rotation)
- False-Open Rate (FAR) optimization
- Temperature scaling calibration
- Float16 & Int8 quantization and export

In [1]:
import sys, os
from pathlib import Path
_root = Path(os.path.abspath('.')).parent if Path(os.path.abspath('.')).name == 'ml' else Path(os.path.abspath('.'))
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
import sys
from pathlib import Path
import os
from pathlib import Path
import numpy as np
import tensorflow as tf
from PIL import Image

from ml.src.dataset import load_dataset_from_directory, split_by_subject, verify_no_subject_leakage
from ml.src.models import build_m1_eye_cnn, build_m2_mobilenet, build_m3_separable
from ml.src.augmentations import EyeAugmentor
from ml.src.metrics import compute_classification_metrics, TemperatureScaler
from ml.src.export import export_to_tflite, verify_tflite_parity

print("TensorFlow Version:", tf.__version__)
print("GPU Devices:", tf.config.list_physical_devices('GPU'))

I0000 00:00:1793293304.851099   52454 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793293304.905190   52454 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1793293309.040371   52454 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


TensorFlow Version: 2.21.0
GPU Devices: []


E0000 00:00:1793293313.946750   52454 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1793293313.973361   52454 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [2]:
# 1. Load and Partition Data with Person-Level Invariant
data_dir = _root / "ml/mock_data"
all_samples = load_dataset_from_directory(data_dir)

# Designate held-out test subjects (never seen during training)
mrl_test_subs = {"s0009", "s0010"}
phone_test_subs = {"phone_user_05"}
test_subject_ids = mrl_test_subs.union(phone_test_subs)

train_samples, val_samples, test_samples = split_by_subject(
    all_samples, test_subject_ids=test_subject_ids, val_ratio=0.15, seed=42
)

verify_no_subject_leakage(train_samples, val_samples, test_samples)
print(f"Partitions: {len(train_samples)} Train | {len(val_samples)} Val | {len(test_samples)} Test")

Partitions: 145 Train | 25 Val | 20 Test


In [3]:
# 2. Load arrays and apply augmentations
augmentor = EyeAugmentor(rng_seed=42)

def samples_to_numpy(sample_list, augment=False):
    X, y = [], []
    for s in sample_list:
        img = Image.open(s.path).convert("L").resize((48, 48))
        arr = np.array(img, dtype=np.float32) / 255.0
        if augment:
            arr = augmentor(arr)
        X.append(np.expand_dims(arr, axis=-1))
        y.append(s.label)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32).reshape(-1, 1)

X_train, y_train = samples_to_numpy(train_samples, augment=True)
X_val, y_val = samples_to_numpy(val_samples, augment=False)
X_test, y_test = samples_to_numpy(test_samples, augment=False)

print(f"X_train shape: {X_train.shape}, Class balance: {np.mean(y_train):.2f} open")

X_train shape: (145, 48, 48, 1), Class balance: 0.49 open


In [4]:
# 3. Build & Train Model M1 (From-Scratch 4-Block CNN)
m1_model = build_m1_eye_cnn(input_shape=(48, 48, 1))
print(f"M1 Parameter Count: {m1_model.count_params():,} (< 100k budget)")

m1_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = m1_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=16,
    verbose=1
)

M1 Parameter Count: 60,817 (< 100k budget)
Epoch 1/10


 1/10 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - accuracy: 0.5000 - loss: 1.1165

 4/10 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.5625 - loss: 0.9264

 7/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.5804 - loss: 0.8315

 9/10 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6042 - loss: 0.7745

...

 1/10 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.9375 - loss: 0.1888

 4/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9375 - loss: 0.1997

 6/10 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9375 - loss: 0.1911

 9/10 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9514 - loss: 0.1730

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 0.9448 - loss: 0.1767 - val_accuracy: 0.4400 - val_loss: 2.0924


In [5]:
# 4. Evaluate Metrics on Held-out Test Sets (T1, T2, T3)
y_pred_prob = m1_model.predict(X_test).flatten()
metrics = compute_classification_metrics(y_test.flatten(), y_pred_prob, threshold=0.5)

print("=== Held-Out Test Evaluation ===")
print(f"Accuracy:         {metrics['accuracy']*100:.2f}%")
print(f"ROC-AUC:          {metrics['roc_auc']:.4f}")
print(f"F1 Score:         {metrics['f1']:.4f}")
print(f"False-Open Rate:  {metrics['false_open_rate']*100:.2f}% (Critical Alarm Metric)")
print(f"False-Closed Rate:{metrics['false_closed_rate']*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step


=== Held-Out Test Evaluation ===
Accuracy:         50.00%
ROC-AUC:          1.0000
F1 Score:         0.0000
False-Open Rate:  0.00% (Critical Alarm Metric)
False-Closed Rate:100.00%


In [6]:
# 5. Export to TFLite (Float16) and Verify Parity
out_tflite = _root / "assets/models/eye_cnn_v1.tflite"
export_meta = export_to_tflite(m1_model, out_tflite, quantization="float16")
print(f"Exported TFLite Size: {export_meta['size_kb']:.2f} KB (< 400 KB budget)")

parity = verify_tflite_parity(m1_model, out_tflite, X_test[:10])
print(f"Parity Max Absolute Diff: {parity['max_abs_diff']:.6f} (Passed: {parity['is_parity_ok']})")

INFO:tensorflow:Assets written to: /tmp/tmpexyyukfq/assets


INFO:tensorflow:Assets written to: /tmp/tmpexyyukfq/assets


Saved artifact at '/tmp/tmpexyyukfq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 48, 48, 1), dtype=tf.float32, name='eye_input')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  140246081721296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140246081721872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140246081721680: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140246081722256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140246081721104: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140246081722640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140246081721488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140246081722448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140246081719952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140246081723600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  140246081722832: Te

W0000 00:00:1793293325.495734   52454 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1793293325.495757   52454 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1793293325.496073   52454 reader.cc:83] Reading SavedModel from: /tmp/tmpexyyukfq
I0000 00:00:1793293325.497157   52454 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1793293325.497174   52454 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpexyyukfq
I0000 00:00:1793293325.541037   52454 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1793293325.542329   52454 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1793293325.766726   52454 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpexyyukfq
I0000 00:00:1793293325.784284   52454 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 288223 microseconds.
I0000 00:00:1793293326.169235   5245

Exported TFLite Size: 124.46 KB (< 400 KB budget)


Parity Max Absolute Diff: 0.000003 (Passed: True)


/home/goharabbas/miniconda3/lib/python3.13/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
